In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
CLEANED_DATA = PROJECT_ROOT / "cleaned_data"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"

team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")

saved_stats = pd.read_csv(PROCESSED_DATA / "team_stats.csv")
results = pd.read_csv(CLEANED_DATA / "results_2000_onwards.csv")

In [2]:
# Iterating thorugh tournaments to see how many unique tournaments are there
for t, c in results["tournament"].value_counts().items():
    print(t, c)
    

Friendly 8423
FIFA World Cup qualification 5840
UEFA Euro qualification 1532
African Cup of Nations qualification 1416
UEFA Nations League 658
AFC Asian Cup qualification 530
African Cup of Nations 525
FIFA World Cup 460
CONCACAF Nations League 422
Gold Cup 359
CECAFA Cup 341
COSAFA Cup 326
CFU Caribbean Cup qualification 293
Island Games 283
UEFA Euro 277
AFC Asian Cup 256
AFF Championship 251
Copa América 248
Gulf Cup 189
SAFF Cup 135
EAFF Championship 130
WAFF Championship 114
UNCAF Cup 109
CONIFA World Football Cup 101
Oceania Nations Cup 99
CFU Caribbean Cup 98
Confederations Cup 96
Arab Cup 92
AFC Challenge Cup qualification 92
AFC Challenge Cup 87
Gold Cup qualification 81
King's Cup 80
Pacific Games 75
Indian Ocean Island Games 68
CONCACAF Nations League qualification 68
Viva World Cup 60
FIFA Series 57
AFF Championship qualification 56
Cyprus International Tournament 54
CONIFA European Football Cup 49
South Pacific Games 48
Muratti Vase 46
Kirin Cup 43
Coupe de l'Outre-Mer 42


In [3]:
# same thing as above.
results["tournament"].nunique()

127

In [4]:
# checking top rows.
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,04-01-2000,Egypt,Togo,2,1,Friendly,Aswan,Egypt,False
1,07-01-2000,Tunisia,Togo,7,0,Friendly,Tunis,Tunisia,False
2,08-01-2000,Trinidad and Tobago,Canada,0,0,Friendly,Port of Spain,Trinidad and Tobago,False
3,09-01-2000,Burkina Faso,Gabon,1,1,Friendly,Ouagadougou,Burkina Faso,False
4,09-01-2000,Guatemala,Armenia,1,1,Friendly,Los Angeles,United States,True


In [5]:
# Checked the dtype of date and found that its a string dtype.
print(results["date"].dtype)

str


In [6]:
# converted to Datetime[us] with dayfirst as True.
results["date"] = pd.to_datetime(results["date"], dayfirst = True)

In [7]:
# result_played will only store the matches which are already played. Future matches are removed (until they happen).
result_played = results[results["home_score"].notna() & results["away_score"].notna()].copy()

In [8]:
# sorted result_played date column to asc.
result_played = result_played.sort_values("date")

In [9]:
# Just checking the latest matches that were added in the data, accurate as of 22-06-2026.
result_played.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
25380,2026-06-27,Panama,England,0,2,FIFA World Cup,East Rutherford,United States,True
25382,2026-06-28,South Africa,Canada,0,1,FIFA World Cup,Inglewood,United States,True
25384,2026-06-29,Germany,Paraguay,1,1,FIFA World Cup,Foxborough,United States,True
25383,2026-06-29,Brazil,Japan,2,1,FIFA World Cup,Houston,United States,True
25385,2026-06-29,Netherlands,Morocco,1,1,FIFA World Cup,Guadalupe,Mexico,True


In [10]:
# checked how many matches since year 2000.
print("result_played: ", len(result_played))
print()

print(result_played[["date", "home_team", "away_team", "home_score", "away_score"]].head())

result_played:  25386

        date            home_team away_team  home_score  away_score
0 2000-01-04                Egypt      Togo           2           1
1 2000-01-07              Tunisia      Togo           7           0
2 2000-01-08  Trinidad and Tobago    Canada           0           0
3 2000-01-09         Burkina Faso     Gabon           1           1
4 2000-01-09            Guatemala   Armenia           1           1


In [11]:
# creaing an ELO System with 1500 as default and K value with 20 points, from which we'll calculate future matche results
# also created a elo rating dict which will store the results.
INITIAL_ELO = 1500
K = 20

elo_ratings = {}

# Created a get_rating function which will get the elo ratings of a team 

def get_rating(team):
    if team not in elo_ratings:
        elo_ratings[team] = INITIAL_ELO
    return elo_ratings[team]

# Created a new get_result function which will store the results Win, Draw, Lose, i.e. W, D, L based on the home-away scores.

def get_result(goals_for, goals_against):
    if goals_for > goals_against:
        return "W"
    elif goals_for < goals_against:
        return "L"
    else:
        return "D"

In [12]:
# Checking to see the first match details are correct.
first_match = result_played.iloc[0]

first_match

date          2000-01-04 00:00:00
home_team                   Egypt
away_team                    Togo
home_score                      2
away_score                      1
tournament               Friendly
city                        Aswan
country                     Egypt
neutral                     False
Name: 0, dtype: object

In [13]:
# Created new Variables that will store home_team, away_team, home_score and away_score.

home_team = first_match["home_team"]
away_team = first_match["away_team"]

home_score = first_match["home_score"]
away_score = first_match["away_score"]

print(home_team, home_score)
print(away_team, away_score)

Egypt 2
Togo 1


In [14]:
# Using the get_rating function which we created above to give both of the teams a default elo rating of 1500.

home_rating = get_rating(home_team)
away_rating = get_rating(away_team)

print(home_rating)
print(away_rating)

1500
1500


In [15]:
# Calculation if teams draw a match.

expected_home = 1 / (
    1 + 10 ** ((away_rating - home_rating) / 400)
)

expected_away = 1 - expected_home

print(expected_home)
print(expected_away)

0.5
0.5


In [16]:
# Giving points based on match results. if a team Wins it gets 1 point, a Lose gets 0 and 0.5 for a Draw.

if home_score > away_score:
    actual_home = 1
    actual_away = 0

elif home_score < away_score:
    actual_home = 0
    actual_away = 1

else:
    actual_home = 0.5
    actual_away = 0.5

print(actual_home)
print(actual_away)

1
0


In [17]:
# Calculating home rating (1500) with K (20) multiplying it by the actual and expected home results and vice versa.
# This will update the current Elo rating of both teams based on their result.

new_home = home_rating + K * (actual_home - expected_home)

new_away = away_rating + K * (actual_away - expected_away)

print(new_home)
print(new_away)

1510.0
1490.0


In [18]:
# store match-by-match elo history 
history_records = []

In [19]:
# Processing matches in chronological order.

for _, match in result_played.iterrows():

    # Match details
    date = match["date"]
    home_team = match["home_team"]
    away_team = match["away_team"]
    tournament = match["tournament"]

    home_score = match["home_score"]
    away_score = match["away_score"]

    # Current ratings
    home_rating = get_rating(home_team)
    away_rating = get_rating(away_team)

    # Expected scores
    expected_home = 1 / (
        1 + 10 ** ((away_rating - home_rating) / 400)
    )
    expected_away = 1 - expected_home

    # Actual scores
    if home_score > away_score:
        actual_home = 1
        actual_away = 0

    elif home_score < away_score:
        actual_home = 0
        actual_away = 1

    else:
        actual_home = 0.5
        actual_away = 0.5

    # New ratings
    new_home_rating = (
        home_rating +
        K * (actual_home - expected_home)
    )

    new_away_rating = (
        away_rating +
        K * (actual_away - expected_away)
    )

    # Results
    home_result = get_result(home_score, away_score)
    away_result = get_result(away_score, home_score)

    # Home team history row
    history_records.append({
        "date": date,
        "team": home_team,
        "opponent": away_team,
        "tournament": tournament,
        "goals_for": home_score,
        "goals_against": away_score,
        "result": home_result,
        "elo_before": home_rating,
        "elo_after": new_home_rating
    })

    # Away team history row
    history_records.append({
        "date": date,
        "team": away_team,
        "opponent": home_team,
        "tournament": tournament,
        "goals_for": away_score,
        "goals_against": home_score,
        "result": away_result,
        "elo_before": away_rating,
        "elo_after": new_away_rating
    })

    # Update ratings dictionary
    elo_ratings[home_team] = new_home_rating
    elo_ratings[away_team] = new_away_rating

In [20]:
# Just checking if history_records is accurate
print(len(history_records))

50772


In [21]:
# Making a new DF out of the history_records.
elo_history = pd.DataFrame(history_records)

In [22]:
# checking if elo_ratings is accurate.
print(len(elo_ratings))

321


In [23]:
# creating new DF from elo_ratings_df that will store only date team and elo.

elo_ratings_df = pd.DataFrame(
    elo_ratings.items(),
    columns=["team", "elo"]
)

In [24]:
# Sorting by highest to lowest.

elo_ratings_df = (
    elo_ratings_df
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)

In [25]:
# 90% accurate to real world ratings
elo_ratings_df.head(5)

,team,elo
0,Argentina,1957.837275
1,Spain,1933.272905
2,France,1907.623943
3,Brazil,1893.649674
4,England,1857.359576


In [26]:
# Round history
elo_history["elo_before"] = elo_history["elo_before"].round(2)
elo_history["elo_after"] = elo_history["elo_after"].round(2)

# Round final ratings
elo_ratings_df["elo"] = elo_ratings_df["elo"].round(2)

# Add ranking
elo_ratings_df.insert(
    0,
    "rank",
    range(1, len(elo_ratings_df) + 1)
)

In [27]:
# Checking if ranking is added succesfully.
elo_ratings_df.head()

,rank,team,elo
0,1,Argentina,1957.84
1,2,Spain,1933.27
2,3,France,1907.62
3,4,Brazil,1893.65
4,5,England,1857.36


In [28]:
# Saving elo_history csv which will contain match by match results and Elo ratings.
elo_history.to_csv(
    PROCESSED_DATA / "elo_history.csv",
    index=False
)

# Saving elo_ratings csv which will contain updated Elo ratings and Rank.
elo_ratings_df.to_csv(
    PROCESSED_DATA / "elo_ratings.csv",
    index=False
)



In [29]:
print(elo_history.shape)
print(elo_ratings_df.shape)

(50772, 9)
(321, 3)


## Elo V1 Outputs

elo_history.csv
- Rows: 50,710
- Columns: 9

elo_ratings.csv
- Rows: 321
- Columns: 3

Parameters:
- Initial Elo = 1500
- K = 20
- No tournament weights
- No home advantage
- No goal difference multiplier

the matches will be updated daily so row counts may vary.